# Подбор моделей по валидации

Первый профиль сравнивает MLP и MGS-KAN на Airfoil: две ступени × две модели × три пары seed на фолде 0. На каждом обучении Adam выполняет 100 эпох. Чекпоинт берётся с минимальным RMSE на валидации. Тест отложен до отдельного решения о финальных конфигурациях.

Для первого запуска на сервере установи проект в окружение ядра с extra `tracking`, укажи `MLFLOW_TRACKING_URI` в корневом `.env` и выбери доступную GPU. Чтобы скачать отсутствующие данные, поставь `DOWNLOAD = True`.

In [ ]:
PROJECT_DIR = None
PROFILE = "airfoil_first"  # airfoil_first (12), airfoil_all (18), deep (54), features
FEATURE_DATASETS = ("airfoil", "power_plant")  # Для всех задач профиля features укажи None.
DEVICE = "cuda:0"
VERSION = "v1"
DOWNLOAD = False

In [ ]:
import sys
from pathlib import Path

if PROJECT_DIR is None:
    cwd = Path.cwd().resolve()
    checkout = next(
        (folder for folder in (cwd, *cwd.parents) if (folder / "src/experiments/runner.py").is_file()),
        None,
    )
else:
    checkout = Path(PROJECT_DIR).expanduser()
    if not checkout.is_absolute() or not (checkout / "src/experiments/runner.py").is_file():
        raise ValueError("Укажи в PROJECT_DIR абсолютный путь к корню mgs-kan на сервере.")
    checkout = checkout.resolve()

if checkout is not None and sys.path[0:1] != [str(checkout)]:
    sys.path.insert(0, str(checkout))

try:
    from src.paths import DATA_DIR, ENV_FILE, PROJECT_ROOT
except ModuleNotFoundError as error:
    raise ImportError("Укажи PROJECT_DIR и выполни ячейки заново.") from error

if checkout is not None and PROJECT_ROOT != checkout:
    raise RuntimeError("Ядро уже загрузило src из другой папки. Перезапусти ядро.")
if not (PROJECT_ROOT / "notebooks/baselines/run_series.ipynb").is_file():
    raise RuntimeError("Укажи PROJECT_DIR с рабочей копией mgs-kan и перезапусти ядро.")
print(f"Проект: {PROJECT_ROOT}\nНастройки: {ENV_FILE}\nДанные: {DATA_DIR}")

## Сетка

`airfoil_all` добавляет третью ступень. При прежних данных, версии и протоколе первые 12 результатов восстанавливаются из MLflow. `deep` запускает глубокий MLP и табличный ResNet ширины 128 на фолде 0 каждой из девяти задач. Занимается одна выбранная GPU, обучения идут последовательно.

В профиле `features` каждая из 12 архитектурных конфигураций обучается на каждом наборе столбцов. Для Airfoil и Power Plant перебираются все непустые наборы. Для остальных задач используются полный набор и предметные группы. Размер MLP подбирается под бюджет MGS-KAN для каждого числа входов.

In [ ]:
from src.config import MonitorSettings, ResourceSettings, TrackingSettings
from src.experiments.search import ValidationGrid

grid = ValidationGrid.preset(PROFILE, datasets=FEATURE_DATASETS if PROFILE == "features" else None)
resources = ResourceSettings(_env_file=ENV_FILE)
print(f"{grid.size} обучений | {grid.training.epochs} эпох | {DEVICE}")
print(f"Датасеты: {', '.join(grid.datasets)} | Фолды: {grid.folds} | Seed: {grid.seeds}")
print(f"Потоков CPU: {resources.cpu_threads} | Доля памяти GPU: {resources.gpu_memory_fraction:.0%}")
for name in grid.datasets:
    print(name, "наборов признаков:", len(grid.subsets(name)))
for stage in grid.stages:
    print(stage.name, *stage.models)

In [ ]:
runs = grid.run(
    version=VERSION, device=DEVICE, root=DATA_DIR, download=DOWNLOAD,
    resources=resources, tracking=TrackingSettings(_env_file=ENV_FILE),
    monitoring=MonitorSettings(_env_file=ENV_FILE),
)
display(runs)

## Выбор конфигурации

Внутри семейства большая ступень выбирается при снижении среднего валидационного RMSE минимум на 1% и улучшении на большинстве пар seed. Смотри `best_epoch` в таблице запусков: если минимум часто близок к последней эпохе, сравни кандидатов с увеличенным лимитом и новой `VERSION`. Значения RMSE сравниваются внутри одного датасета и фолда.

In [ ]:
comparison = grid.compare(runs)
display(comparison.sort_values(["dataset", "fold", "model", "parameters"]))
display(comparison.loc[comparison.selected].sort_values(["dataset", "fold", "mean_val_rmse"]))

if PROFILE == "features":
    selected = comparison.loc[comparison.selected].sort_values(["mean_val_rmse", "n_features", "parameters"])
    display(selected.groupby(["dataset", "fold", "model"], as_index=False).head(1))
    by_size, paired = grid.feature_reports(runs)
    display(by_size)
    display(paired)  # ΔRMSE = MLP − MGS-KAN; положительная разница в пользу MGS-KAN.